# Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pointbiserialr, chi2_contingency
from sklearn.preprocessing import StandardScaler

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

print("✅ Libraries imported successfully")

# Load your data

In [ ]:
df = pd.read_csv('your_stock_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f"Data loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

# Create Labels Function

In [ ]:
def create_labels_with_features(df, window=30, threshold=0.003):
    """
    Create labels and ensure they align with feature rows
    """
    df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
    
    labels = []
    valid_indices = []
    
    for ticker, group in df.groupby('Ticker'):
        group = group.reset_index(drop=True)
        prices = group["Close"].values
        
        for i in range(len(prices) - window - 1):
            ret = (prices[i+window] - prices[i+window-1]) / prices[i+window-1]
            
            if ret > threshold:
                label = 1   # up
            elif ret < -threshold:
                label = 0   # down
            else:
                continue    # flat
            
            labels.append(label)
            valid_indices.append(group.index[i + window - 1])
    
    # Create labeled dataframe
    df_labeled = df.loc[valid_indices].copy()
    df_labeled['label'] = labels
    
    return df_labeled

print("✅ Label creation function defined")

#  Generate Labels

In [ ]:
# Create labels
window = 30
threshold = 0.003

df_labeled = create_labels_with_features(df, window=window, threshold=threshold)

print(f"Labeled data shape: {df_labeled.shape}")
print(f"Up (1): {(df_labeled['label']==1).sum()}")
print(f"Down (0): {(df_labeled['label']==0).sum()}")
print(f"Up percentage: {(df_labeled['label']==1).sum() / len(df_labeled) * 100:.2f}%")

df_labeled.head()

# Define Feature Columns

In [ ]:
# Define features to analyze
feature_cols = [
    'Open', 'High', 'Low', 'Volume',
]

print(f"Analyzing {len(feature_cols)} features:")
for i, feat in enumerate(feature_cols, 1):
    print(f"  {i}. {feat}")

# Plot 1 - Correlation Bar Chart

In [ ]:
def plot_feature_target_correlation(df, feature_cols, target_col='label'):
    """
    Plot correlation between features and binary target
    """
    correlations = []
    p_values = []
    
    for feature in feature_cols:
        valid_data = df[[feature, target_col]].dropna()
        
        if len(valid_data) > 0:
            corr, p_val = pointbiserialr(valid_data[target_col], valid_data[feature])
            correlations.append(corr)
            p_values.append(p_val)
        else:
            correlations.append(0)
            p_values.append(1)
    
    # Create DataFrame
    corr_df = pd.DataFrame({
        'Feature': feature_cols,
        'Correlation': correlations,
        'P_value': p_values,
        'Significant': ['***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '' 
                       for p in p_values]
    })
    
    corr_df['Abs_Correlation'] = corr_df['Correlation'].abs()
    corr_df = corr_df.sort_values('Abs_Correlation', ascending=True)
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 10))
    
    # Left: Correlation values
    colors = ['red' if x < 0 else 'green' for x in corr_df['Correlation']]
    ax1.barh(corr_df['Feature'], corr_df['Correlation'], color=colors, alpha=0.7)
    
    for i, (idx, row) in enumerate(corr_df.iterrows()):
        if row['Significant']:
            ax1.text(row['Correlation'], i, f" {row['Significant']}", 
                    va='center', fontsize=10, fontweight='bold')
    
    ax1.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax1.set_xlabel('Point-Biserial Correlation', fontsize=12)
    ax1.set_title('Feature Correlation with Target Label\n(Green=Up, Red=Down)', fontsize=14)
    ax1.grid(axis='x', alpha=0.3)
    
    # Right: Absolute correlation
    ax2.barh(corr_df['Feature'], corr_df['Abs_Correlation'], 
             color='steelblue', alpha=0.7)
    ax2.set_xlabel('Absolute Correlation (Strength)', fontsize=12)
    ax2.set_title('Feature Importance', fontsize=14)
    ax2.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('feature_target_correlation.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*80)
    print("FEATURE CORRELATION WITH TARGET LABEL")
    print("="*80)
    print(corr_df[['Feature', 'Correlation', 'P_value', 'Significant']].to_string(index=False))
    
    return corr_df

# Run correlation analysis
correlation_results = plot_feature_target_correlation(df_labeled, feature_cols)

# Plot 2 - Distribution Plots

In [ ]:
def plot_feature_distributions_by_label(df, feature_cols, target_col='label'):
    """
    Plot feature distributions split by up/down label
    """
    n_features = len(feature_cols)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5*n_rows))
    axes = axes.flatten()
    
    for idx, feature in enumerate(feature_cols):
        ax = axes[idx]
        
        down_data = df[df[target_col] == 0][feature].dropna()
        up_data = df[df[target_col] == 1][feature].dropna()
        
        if len(down_data) > 0:
            ax.hist(down_data, bins=50, alpha=0.5, label='Down (0)', 
                   color='red', density=True)
        if len(up_data) > 0:
            ax.hist(up_data, bins=50, alpha=0.5, label='Up (1)', 
                   color='green', density=True)
        
        if len(down_data) > 0:
            ax.axvline(down_data.mean(), color='red', linestyle='--', 
                      linewidth=2, alpha=0.7)
        if len(up_data) > 0:
            ax.axvline(up_data.mean(), color='green', linestyle='--', 
                      linewidth=2, alpha=0.7)
        
        ax.set_title(feature, fontsize=11, fontweight='bold')
        ax.set_xlabel('Value', fontsize=9)
        ax.set_ylabel('Density', fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    
    for idx in range(n_features, len(axes)):
        fig.delaxes(axes[idx])
    
    plt.suptitle('Feature Distributions by Target Label (Up vs Down)', 
                 fontsize=16, y=1.00)
    plt.tight_layout()
    plt.savefig('feature_distributions_by_label.png', dpi=300, bbox_inches='tight')
    plt.show()

# Plot distributions
plot_feature_distributions_by_label(df_labeled, feature_cols)